# Model Benchmark — one shared split, one leaderboard

Every model below imports the **same** data from `prep.py`, so they all train on the
identical 80% pool and are scored on the identical frozen 20% test set
(stratified on quantile bins of log-velocity). This is the only way the
MAE / RMSE / R² numbers are truly head-to-head.

**Design**
- `prep.get_data()` returns `X_train`/`X_val` (64/16, for early stopping & tuning),
  `X_trainval` (the full 80%), and the common `X_test`.
- Models that need a validation set (LightGBM, TabNet, GNN) tune/early-stop on
  `X_val`, then **refit on `X_trainval`** so every model's training budget is the
  same 80%.
- Metrics are always computed on **original SV units** via `prep.evaluate` (it
  inverts the log transform automatically if `CONFIG.log_target=True`).

**Saved models.** Each model is written to `models/` at its best epoch/iteration
(ElasticNet/SVR/LightGBM as `.pkl`, TabNet as `.zip` + its scaler, GNN as a `.pt`
checkpoint with edge index, target stats, and scaler) for further analysis.

**Disclosure:** the **GNN is transductive** — its kNN graph spans all nodes using
features only (no labels). Even on an identical split it sees test-node *features*
during training, so it carries a methodological asterisk vs. the inductive models.

Change `prep.CONFIG` (features, log target) in ONE place and rerun; delete
`split_indices.npz` if you change the split parameters.


In [2]:
import importlib, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

os.makedirs("models", exist_ok=True)   # each model is saved here at its best epoch

import prep
importlib.reload(prep)

# ---- set the shared basis here (applies to ALL models) ----
prep.CONFIG.feature_cols = [0]   # e.g. [0,5,9]; or list(range(0,14)) for all
prep.CONFIG.log_target   = False       # try True to predict log10(SV)

d = prep.get_data()
print("Features :", d.feature_names)
print("Target   :", d.target_name, "| log_target =", prep.CONFIG.log_target)
print("Sizes    : train=%d  val=%d  test=%d  (trainval=%d)"
      % (len(d.train_idx), len(d.val_idx), len(d.test_idx), len(d.trainval_idx)))

results = []          # list of metric dicts
test_preds = {}       # name -> predicted TEST values (original units) for plotting/metrics
full_preds = {}       # name -> predicted ALL-particle values (original units) for export

def show(r):          # clean per-model line: MAE, MSE, RMSE, R2
    print(f"{r['model']:20s} | MAE {r['MAE']:.4f} | MSE {r['MSE']:.4f} "
          f"| RMSE {r['RMSE']:.4f} | R2 {r['R2']:.4f}")

Features : ['ESD(um)']
Target   : SettlingVelocity(mpd) | log_target = False
Sizes    : train=1958  val=490  test=613  (trainval=2448)


In [3]:
# ===== Linear Regression / OLS (plain baseline, SAME shared split) =====
# Added so the simple linear baseline is scored on the IDENTICAL test rows as
# every other model. That makes its MAE/MSE directly comparable -- the whole
# "different-split" effect (where a standalone LR showed low MSE just because its
# test set had smaller target variance) disappears once it's on this split.
from sklearn.linear_model import LinearRegression

ols = LinearRegression()
ols.fit(d.X_trainval, d.y_trainval)              # train on the shared 80%
pred = ols.predict(d.X_test)                     # score the shared frozen test set
results.append(prep.evaluate("LinearRegression", d.y_test_orig, pred))
test_preds["LinearRegression"] = prep.inverse(pred)
joblib.dump(ols, "models/LinearRegression.pkl")
show(results[-1])

LinearRegression     | MAE 10.9205 | MSE 1063.3559 | RMSE 32.6091 | R2 0.2761


In [4]:
# ===== ElasticNet (linear, no val set needed -> train on full 80%) =====
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV

enet = Pipeline([("sc", StandardScaler()),
                 ("m", ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,.99,1.0],
                                    alphas=np.logspace(-3,1,100), cv=5,
                                    max_iter=10000, random_state=prep.CONFIG.seed))])
enet.fit(d.X_trainval, d.y_trainval)
pred = enet.predict(d.X_test)
results.append(prep.evaluate("ElasticNet", d.y_test_orig, pred))
test_preds["ElasticNet"] = prep.inverse(pred)
joblib.dump(enet, "models/ElasticNet.pkl")
show(results[-1])

ElasticNet           | MAE 10.9025 | MSE 1064.0163 | RMSE 32.6193 | R2 0.2757


In [5]:
# ===== SVR (kernel, scale inside pipeline, tune on 80% via CV) =====
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

# cache_size bumps the kernel cache (MB) so high-feature fits don't thrash;
# max_iter caps each fit so a non-converging combo can't hang the whole search.
svr = Pipeline([("sc", StandardScaler()),
                ("m", SVR(cache_size=500, max_iter=200000))])
# Wider grid + refit on R2 (not MAE) so the SVR is tuned on the same
# objective we use to rank every model on the shared leaderboard.
# NOTE: the poly kernel was dropped -- with C>=100 / gamma=1 it frequently
# fails to converge (SVR's default max_iter=-1 means it can run effectively
# forever), which is what made this cell appear to hang, especially once
# more features were selected. rbf alone is the usual winner on tabular data.
grid = [
    {"m__kernel": ["rbf"], "m__C": [0.1, 1, 10, 100],
     "m__epsilon": [0.01, 0.1, 0.2, 0.5], "m__gamma": ["scale", "auto", 0.01, 0.1, 1]},
    {"m__kernel": ["linear"], "m__C": [0.1, 1, 10, 100],
     "m__epsilon": [0.01, 0.1, 0.2, 0.5]},
]
gs = GridSearchCV(svr, grid, cv=5, scoring="r2", refit=True, n_jobs=-1)
gs.fit(d.X_trainval, d.y_trainval)
pred = gs.predict(d.X_test)
results.append(prep.evaluate("SVR", d.y_test_orig, pred, note=str(gs.best_params_)))
test_preds["SVR"] = prep.inverse(pred)
joblib.dump(gs.best_estimator_, "models/SVR.pkl")
show(results[-1])

SVR                  | MAE 8.1402 | MSE 1051.5890 | RMSE 32.4282 | R2 0.2841


In [6]:
# ===== LightGBM (tune on train, early-stop on val, refit on 80%) =====
try:
    from lightgbm import LGBMRegressor, early_stopping
    from sklearn.model_selection import RandomizedSearchCV

    # Lean search: ~6 configs x 3-fold = 18 fits of <=600 trees (was 40 x 5 x 2000).
    param_dist = {"learning_rate":[0.05, 0.1], "num_leaves":[15, 31],
                  "max_depth":[-1, 5], "min_child_samples":[10, 20],
                  "colsample_bytree":[0.8, 1.0]}
    # n_jobs=1 on the LightGBM estimator itself: RandomizedSearchCV already
    # parallelizes across folds/configs, so letting each LightGBM fit also
    # grab all cores causes thread oversubscription (very slow on Windows).
    base = LGBMRegressor(objective="regression", n_estimators=600, n_jobs=1,
                         random_state=prep.CONFIG.seed, verbose=-1)
    search = RandomizedSearchCV(base, param_dist, n_iter=6, cv=3,
                                scoring="r2",   # tune for R2, same objective as the leaderboard
                                random_state=prep.CONFIG.seed, n_jobs=-1)
    search.fit(d.X_train, d.y_train)               # tune on 64%
    bp = search.best_params_

    cap = LGBMRegressor(objective="regression", n_estimators=600,
                        random_state=prep.CONFIG.seed, verbose=-1, **bp)
    cap.fit(d.X_train, d.y_train, eval_set=[(d.X_val, d.y_val)], eval_metric="l2",
            callbacks=[early_stopping(30)])        # val sets best_iteration_
    n_best = cap.best_iteration_ or 600

    final = LGBMRegressor(objective="regression", n_estimators=n_best,
                          random_state=prep.CONFIG.seed, verbose=-1, **bp)
    final.fit(d.X_trainval, d.y_trainval)          # refit on 80% at best n_estimators
    pred = final.predict(d.X_test)
    results.append(prep.evaluate("LightGBM", d.y_test_orig, pred,
                                 note=f"n_estimators={n_best}"))
    test_preds["LightGBM"] = prep.inverse(pred)
    joblib.dump(final, "models/LightGBM.pkl")      # saved at best iteration
    show(results[-1])
except ImportError:
    print("LightGBM not installed - skipping (pip install lightgbm).")

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[10]	valid_0's l2: 640.358
LightGBM             | MAE 9.5198 | MSE 1144.9508 | RMSE 33.8371 | R2 0.2206


In [7]:
# ===== GNN GraphSAGE (transductive; refit on 80%) =====
try:
    import torch, torch.nn as nn
    from sklearn.neighbors import kneighbors_graph
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv

    torch.manual_seed(prep.CONFIG.seed)
    # scale features on trainval; standardize target on trainval
    sc = prep.scale(d.X_trainval, d.X_all)[0]
    Xs = torch.tensor(sc.transform(d.X_all), dtype=torch.float32)
    ymu = d.y_trainval.mean(); ysd = d.y_trainval.std() + 1e-8
    yz = torch.tensor(((d.y_all - ymu)/ysd).reshape(-1,1), dtype=torch.float32)

    A = kneighbors_graph(Xs.numpy(), n_neighbors=10, mode="connectivity", include_self=False)
    edge_index = torch.tensor(np.array(A.nonzero()), dtype=torch.long)
    data = Data(x=Xs, edge_index=edge_index, y=yz)

    def mask(idx):
        m = torch.zeros(len(d.df), dtype=torch.bool); m[idx] = True; return m
    m_tr, m_va, m_te = mask(d.train_idx), mask(d.val_idx), mask(d.test_idx)
    m_trva = m_tr | m_va

    class Net(nn.Module):
        def __init__(self, din, dh=64, p=0.3):
            super().__init__()
            self.c1=SAGEConv(din,dh); self.c2=SAGEConv(dh,dh)
            self.do=nn.Dropout(p); self.fc=nn.Linear(dh,1)
        def forward(self, dd):
            x=torch.relu(self.c1(dd.x,dd.edge_index)); x=self.do(x)
            x=torch.relu(self.c2(x,dd.edge_index)); return self.fc(x)

    crit = nn.MSELoss()
    def train(train_mask, epochs, patience=60):
        net=Net(Xs.shape[1]); opt=torch.optim.Adam(net.parameters(),lr=0.01,weight_decay=5e-4)
        sch=torch.optim.lr_scheduler.StepLR(opt,step_size=100,gamma=0.5)
        best=1e9; best_state=None; best_ep=0
        for ep in range(epochs):
            net.train(); opt.zero_grad()
            loss=crit(net(data)[train_mask], data.y[train_mask])
            loss.backward(); opt.step(); sch.step()
            net.eval()
            with torch.no_grad():
                v=crit(net(data)[m_va], data.y[m_va]).item()
            if v<best: best,best_ep=v,ep; best_state={k:val.clone() for k,val in net.state_dict().items()}
        net.load_state_dict(best_state); return net, best_ep
    _, best_ep = train(m_tr, 400)                  # find best epoch on val
    net, _ = train(m_trva, best_ep+1)              # refit on 80%

    net.eval()
    with torch.no_grad():
        out = net(data).cpu().numpy().ravel()
    pred_all = out*ysd + ymu                        # back to model space
    pred = pred_all[d.test_idx]
    results.append(prep.evaluate("GNN (transductive)", d.y_test_orig, pred,
                                 note=f"epochs={best_ep+1}; transductive"))
    test_preds["GNN (transductive)"] = prep.inverse(pred)
    # save the refit (best-epoch) model + the bits needed to reuse it
    torch.save({"state_dict": net.state_dict(), "edge_index": edge_index,
                "y_mean": ymu, "y_std": ysd, "best_epoch": best_ep + 1,
                "feature_cols": prep.CONFIG.feature_cols,
                "scaler": sc}, "models/GNN.pt")
    show(results[-1])
except ImportError:
    print("torch_geometric / torch not installed - skipping.")

GNN (transductive)   | MAE 9.4052 | MSE 996.2484 | RMSE 31.5634 | R2 0.3218


In [8]:
# ===== TabNet (early-stop on val, refit on 80%) =====
try:
    import torch
    from pytorch_tabnet.tab_model import TabNetRegressor

    tabnet_scaler, Xtr, Xva, Xte, Xtrva = prep.scale(d.X_trainval, d.X_train, d.X_val,
                                          d.X_test, d.X_trainval)
    ytr = d.y_train.reshape(-1,1); yva = d.y_val.reshape(-1,1)
    ytrva = d.y_trainval.reshape(-1,1)

    def new_tabnet():
        return TabNetRegressor(n_d=16, n_a=16, n_steps=3, seed=prep.CONFIG.seed,
                               optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
                               scheduler_params=dict(step_size=50, gamma=0.9),
                               scheduler_fn=torch.optim.lr_scheduler.StepLR)
    m = new_tabnet()
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_name=["val"], eval_metric=["rmse"],
          max_epochs=200, patience=30, batch_size=256, virtual_batch_size=128)
    best_epoch = int(np.argmin(m.history["val_rmse"])) + 1

    mf = new_tabnet()                              # refit on 80% for best_epoch
    mf.fit(Xtrva, ytrva, max_epochs=best_epoch, batch_size=256, virtual_batch_size=128)
    pred = mf.predict(Xte).ravel()
    results.append(prep.evaluate("TabNet", d.y_test_orig, pred,
                                 note=f"epochs={best_epoch}"))
    test_preds["TabNet"] = prep.inverse(pred)
    mf.save_model("models/TabNet")                 # -> models/TabNet.zip (best epoch)
    joblib.dump(tabnet_scaler, "models/TabNet_scaler.pkl")    # scaler needed to reuse it
    show(results[-1])
except ImportError:
    print("pytorch-tabnet / torch not installed - skipping.")

/home/hhuan006/envs/SVML/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 798.38534| val_rmse: 26.82447|  0:00:00s
epoch 1  | loss: 655.5087| val_rmse: 27.03135|  0:00:01s
epoch 2  | loss: 596.84893| val_rmse: 26.88207|  0:00:01s
epoch 3  | loss: 630.0162| val_rmse: 25.48773|  0:00:01s
epoch 4  | loss: 597.81671| val_rmse: 25.20507|  0:00:02s
epoch 5  | loss: 596.06394| val_rmse: 25.31379|  0:00:02s
epoch 6  | loss: 582.23745| val_rmse: 25.31073|  0:00:03s
epoch 7  | loss: 604.2162| val_rmse: 25.36084|  0:00:03s
epoch 8  | loss: 607.15051| val_rmse: 25.48195|  0:00:03s
epoch 9  | loss: 571.87631| val_rmse: 25.44416|  0:00:04s
epoch 10 | loss: 544.39386| val_rmse: 25.48209|  0:00:04s
epoch 11 | loss: 640.09539| val_rmse: 25.55427|  0:00:04s
epoch 12 | loss: 626.1921| val_rmse: 25.57431|  0:00:05s
epoch 13 | loss: 605.73506| val_rmse: 25.20444|  0:00:05s
epoch 14 | loss: 596.68818| val_rmse: 25.16514|  0:00:06s
epoch 15 | loss: 642.03808| val_rmse: 25.45258|  0:00:06s
epoch 16 | loss: 638.20869| val_rmse: 26.04419|  0:00:06s
epoch 17 | loss: 6

/home/hhuan006/envs/SVML/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/home/hhuan006/envs/SVML/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/hhuan006/envs/SVML/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 784.2529|  0:00:00s
epoch 1  | loss: 641.13048|  0:00:00s
epoch 2  | loss: 593.62267|  0:00:01s
epoch 3  | loss: 605.59651|  0:00:01s
epoch 4  | loss: 606.53799|  0:00:02s
epoch 5  | loss: 626.95153|  0:00:02s
epoch 6  | loss: 614.41578|  0:00:03s
epoch 7  | loss: 627.00632|  0:00:03s
epoch 8  | loss: 637.78525|  0:00:04s
epoch 9  | loss: 624.79522|  0:00:04s
epoch 10 | loss: 595.63961|  0:00:05s
epoch 11 | loss: 593.7605|  0:00:05s
epoch 12 | loss: 602.2695|  0:00:06s
epoch 13 | loss: 615.69124|  0:00:06s
epoch 14 | loss: 493.69039|  0:00:07s
epoch 15 | loss: 613.09055|  0:00:07s
epoch 16 | loss: 619.59224|  0:00:07s
epoch 17 | loss: 615.17683|  0:00:08s
epoch 18 | loss: 564.66141|  0:00:08s
epoch 19 | loss: 532.32794|  0:00:09s
epoch 20 | loss: 604.06443|  0:00:09s
epoch 21 | loss: 638.43202|  0:00:10s
epoch 22 | loss: 544.81989|  0:00:10s
epoch 23 | loss: 561.53321|  0:00:11s
epoch 24 | loss: 627.67743|  0:00:11s
epoch 25 | loss: 625.68409|  0:00:12s
epoch 26 | loss

In [9]:
# ===== Permutation importance (model-agnostic, on the shared test set) =====
# Shuffle one feature at a time and measure the R2 drop -- answers "how much
# does this feature actually matter for this model's predictions," for every
# model in the benchmark, including TabNet and the transductive GNN.
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

importance_table = {}

# --- sklearn-compatible models: ElasticNet, SVR, LightGBM -----------------
fitted_models = {}
if "enet" in globals():       fitted_models["ElasticNet"] = enet
if "gs" in globals():         fitted_models["SVR"]        = gs.best_estimator_
if "final" in globals():      fitted_models["LightGBM"]   = final

for name, est in fitted_models.items():
    r = permutation_importance(est, d.X_test, d.y_test, n_repeats=20,
                                scoring="r2", random_state=prep.CONFIG.seed)
    importance_table[name] = pd.Series(r.importances_mean, index=d.feature_names)

# --- TabNet: wrap scaler+model behind a plain .predict(X) so sklearn's
#     permutation_importance can drive it like any other estimator ---------
if "mf" in globals() and "tabnet_scaler" in globals():
    class TabNetWrapper:
        def __init__(self, model, scaler):
            self.model, self.scaler = model, scaler
        def predict(self, X):
            return self.model.predict(self.scaler.transform(X)).ravel()
        def fit(self, X, y=None):   # permutation_importance only needs predict,
            return self              # but some sklearn versions check for fit/get_params
        def get_params(self, deep=True):
            return {}

    tabnet_wrapped = TabNetWrapper(mf, tabnet_scaler)
    r = permutation_importance(tabnet_wrapped, d.X_test, d.y_test, n_repeats=20,
                                scoring="r2", random_state=prep.CONFIG.seed)
    importance_table["TabNet"] = pd.Series(r.importances_mean, index=d.feature_names)

# --- GNN: transductive, so it needs a full-graph forward pass each time.
#     Manually shuffle one feature column among test rows only, rebuild the
#     scaled node-feature matrix, rerun the fixed graph, and compare R2. ----
if "net" in globals() and "edge_index" in globals():
    base_r2 = r2_score(d.y_test, pred)   # `pred` = GNN's own test predictions (model space)
    rng = np.random.RandomState(prep.CONFIG.seed)
    gnn_importances = []
    for c in range(d.X_all.shape[1]):
        drops = []
        for _ in range(20):
            Xp = d.X_all.copy()
            perm = rng.permutation(d.test_idx)
            Xp[d.test_idx, c] = Xp[perm, c]          # shuffle within test rows only
            Xs_p = torch.tensor(sc.transform(Xp), dtype=torch.float32)
            data_p = Data(x=Xs_p, edge_index=edge_index)
            net.eval()
            with torch.no_grad():
                out_p = net(data_p).cpu().numpy().ravel()
            pred_p = out_p[d.test_idx] * ysd + ymu    # back to model space
            drops.append(base_r2 - r2_score(d.y_test, pred_p))
        gnn_importances.append(np.mean(drops))
    importance_table["GNN (transductive)"] = pd.Series(gnn_importances, index=d.feature_names)

imp_df = pd.DataFrame(importance_table).sort_values(
    by=list(importance_table)[0], ascending=False)
imp_df.to_csv("feature_importance.csv")
print("Saved -> feature_importance.csv")
display(imp_df.style.format("{:.4f}"))

Saved -> feature_importance.csv


,ElasticNet,SVR,LightGBM,TabNet,GNN (transductive)
ESD(um),0.4468,0.4213,0.3170,0.5362,-0.0234


In [10]:
# ===== Leaderboard =====
lb = (pd.DataFrame(results)
        .sort_values("R2", ascending=False)
        .reset_index(drop=True)[["model","MAE","RMSE","MSE","R2","note"]])
lb.to_csv("leaderboard.csv", index=False)
print("Saved -> leaderboard.csv")
display(lb.style.format({"MAE":"{:.3f}","RMSE":"{:.3f}","MSE":"{:.2f}","R2":"{:.3f}"}))

Saved -> leaderboard.csv


,model,MAE,RMSE,MSE,R2,note
0,TabNet,9.100,31.356,983.18,0.331,epochs=49
1,GNN (transductive),9.405,31.563,996.25,0.322,epochs=256; transductive
2,SVR,8.140,32.428,1051.59,0.284,"{'m__C': 100, 'm__epsilon': 0.2, 'm__gamma': 0.01, 'm__kernel': 'rbf'}"
3,LinearRegression,10.921,32.609,1063.36,0.276,
4,ElasticNet,10.903,32.619,1064.02,0.276,
5,LightGBM,9.520,33.837,1144.95,0.221,n_estimators=10


In [11]:
# ===== Predicted vs observed on the shared test set =====
names = list(test_preds)
ncol = 3; nrow = int(np.ceil(len(names)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 4.5*nrow), squeeze=False)
true = d.y_test_orig
for ax, name in zip(axes.ravel(), names):
    p = test_preds[name]
    ax.scatter(true, p, alpha=0.4, s=12)
    lims = [min(true.min(), p.min()), max(true.max(), p.max())]
    ax.plot(lims, lims, "r--", lw=1)
    r2 = next(r["R2"] for r in results if r["model"] == name)
    ax.set_title(f"{name}  (R2={r2:.3f})")
    ax.set_xlabel("Observed SV"); ax.set_ylabel("Predicted SV")
for ax in axes.ravel()[len(names):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig("benchmark_pred_vs_obs.png", dpi=200); plt.show()

In [12]:
# ===== Export predictions for ALL particles (for scientific application) =====
# Predicts every cleaned particle with each model, for downstream use (flux,
# maps, etc.).
#
# IMPORTANT: a `split` column flags each row. Only split=="test" rows are
# out-of-sample (honest). train/val predictions are IN-SAMPLE (the model was fit
# on them) and are optimistic -- do NOT compute performance metrics over all rows;
# use the leaderboard (test-only) numbers for performance.

# --- sklearn / deterministic models: predict the full dataset directly ---
sk_models = {}
if "ols"    in globals(): sk_models["LinearRegression"] = ols
if "enet"   in globals(): sk_models["ElasticNet"]       = enet
if "gs"     in globals(): sk_models["SVR"]              = gs.best_estimator_
if "final"  in globals(): sk_models["LightGBM"]         = final
# if "tabpfn" in globals(): sk_models["TabPFN"]           = tabpfn
for name, est in sk_models.items():
    full_preds[name] = prep.inverse(np.asarray(est.predict(d.X_all)).ravel())

# --- TabNet: needs its own scaler applied to all particles, then .predict() ---
if "mf" in globals() and "tabnet_scaler" in globals():
    Xall_sc = tabnet_scaler.transform(d.X_all).astype(np.float32)
    full_preds["TabNet"] = prep.inverse(mf.predict(Xall_sc).ravel())

# --- GNN: transductive -> one forward pass already yields ALL node predictions ---
if "net" in globals() and "data" in globals() and "ymu" in globals() and "ysd" in globals():
    import torch
    net.eval()
    with torch.no_grad():
        out_all = net(data).cpu().numpy().ravel()
    full_preds["GNN (transductive)"] = prep.inverse(out_all * ysd + ymu)

# split labels so in-sample rows are clearly flagged
split = np.array(["train"] * len(d.df), dtype=object)
split[d.val_idx]  = "val"
split[d.test_idx] = "test"

out = pd.DataFrame({"ESD":    d.df.iloc[:, 0].values,
                    "y_true": d.y_all_orig,
                    "split":  split})
for name in test_preds:                       # keep a consistent column order
    if name in full_preds:
        out[f"pred_{name}"] = np.asarray(full_preds[name]).ravel()

out.to_csv("predictions_all_particles.csv", index=False)
print("Saved -> predictions_all_particles.csv", out.shape)
print("Columns:", [c for c in out.columns if c.startswith("pred_")])
print("NOTE: only split=='test' rows are out-of-sample; do NOT compute metrics over all rows.")
out.head()

Saved -> predictions_all_particles.csv (3061, 9)
Columns: ['pred_LinearRegression', 'pred_ElasticNet', 'pred_SVR', 'pred_LightGBM', 'pred_GNN (transductive)', 'pred_TabNet']
NOTE: only split=='test' rows are out-of-sample; do NOT compute metrics over all rows.


,ESD,y_true,split,pred_LinearRegression,pred_ElasticNet,pred_SVR,pred_LightGBM,pred_GNN (transductive),pred_TabNet
0,46.905841,2.917051,test,4.641992,4.672167,4.941069,8.416309,4.884901,7.145582
1,64.938197,5.564851,val,8.775473,8.789099,5.325047,8.502871,5.674703,6.553981
2,59.021889,1.544486,train,7.419303,7.438358,5.148084,8.502871,5.462688,6.713283
3,141.367492,10.762916,val,26.295041,26.238526,12.172876,15.667720,24.271106,20.429140
4,104.884649,2.732544,train,17.932232,17.909198,7.843695,11.843995,8.691518,8.092258
